# 작업형 2유형 — 풀이 골격 (template)

시험장에서 그대로 따라 칠 수 있는 표준 흐름. titanic 생존 예측(ROC-AUC, 확률 제출)로 1회 완주한다.

> **순서**: 데이터 로드 → EDA → 결측치 → 인코딩 → (스케일링) → 검증분할 → 학습/평가 → 예측 → `result.csv`

> train에 한 전처리는 **반드시 test에 똑같이** 적용한다.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

### 0. 데이터 준비 (시험 환경 재현)

실제 시험은 `train.csv`, `test.csv`가 주어진다. 여기선 내장 데이터를 쪼개 흉내 낸다.
`y_test`는 채점용으로만 두고 풀이에는 쓰지 않는다.

In [ ]:
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','sibsp','parch','fare','embarked']]
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df['survived'])
y_test = test['survived']                                   # 채점용(시험엔 없음)
test  = test.drop(columns='survived').reset_index(drop=True)
train = train.reset_index(drop=True)
train.shape, test.shape

### 1. EDA
타입·결측·분포를 먼저 본다.

In [ ]:
train.info()
print(train.isnull().sum())
train.describe(include='all')

### 2. 전처리
target 분리 → 결측치 → 인코딩. train 기준 통계로 test도 채운다.

In [ ]:
target = train['survived']
X = train.drop(columns='survived')

# 수치형 결측 → 중앙값 (train 기준값을 test에도)
for c in ['age','fare']:
    m = X[c].median()
    X[c]  = X[c].fillna(m)
    test[c] = test[c].fillna(m)

# 범주형 결측 → 최빈값
mode_emb = X['embarked'].mode()[0]
X['embarked']    = X['embarked'].fillna(mode_emb)
test['embarked'] = test['embarked'].fillna(mode_emb)

In [ ]:
# 레이블 인코딩 (object 컬럼)
from sklearn.preprocessing import LabelEncoder
for c in ['sex','embarked']:
    le = LabelEncoder()
    X[c]    = le.fit_transform(X[c])
    test[c] = le.transform(test[c])
X.head()

### 3. 검증 데이터 분할
제출 전 성능을 가늠하려면 train 안에서 다시 쪼갠다.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, target, test_size=0.2, random_state=42, stratify=target)

### 4. 학습 및 평가
분류 기본은 RandomForest. 확률 제출이면 `predict_proba`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

val_pred = rf.predict_proba(X_val)[:, 1]
print('검증 ROC-AUC:', roc_auc_score(y_val, val_pred))

**(선택) LightGBM** — 설치돼 있으면 보통 더 강하다. 시험장에 없을 수도 있으니 RF를 기본으로.

In [ ]:
# from lightgbm import LGBMClassifier
# lgb = LGBMClassifier(random_state=42)
# lgb.fit(X_tr, y_tr)
# print('LGBM val AUC:', roc_auc_score(y_val, lgb.predict_proba(X_val)[:,1]))

### 5. 예측 및 제출 파일
문제에서 요구한 컬럼명(`pred`)·파일명(`result.csv`)을 정확히 지킨다.

In [ ]:
pred = rf.predict_proba(test)[:, 1]            # 확률 제출
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
result.head()

In [ ]:
# (참고) 실제 채점 점수 — 시험에선 알 수 없음
from sklearn.metrics import roc_auc_score
print('test ROC-AUC:', roc_auc_score(y_test, pred))

---
### 제출 형식 체크리스트

- [ ] 컬럼명이 문제 지시와 정확히 같은가 (`pred`)
- [ ] 확률/클래스 중 무엇을 요구했는가 (`predict_proba[:,1]` vs `predict`)
- [ ] `index=False` 로 저장했는가
- [ ] 행 개수 = test 행 개수와 같은가